Baseball

In [1]:
# Importing all required packages - I installed pybaseball while working on the 2023 data
import pandas as pd
import numpy as np

from pybaseball import statcast
from pybaseball import playerid_reverse_lookup

In [2]:
#I googled the 2024 baseball season dates, which started in March in South Korea, and ended with the World Series
df = statcast(start_dt="2024-03-15", end_dt="2024-11-02")
df.head()

This is a large query, it may take a moment to complete


100%|██████████| 233/233 [00:20<00:00, 11.14it/s]
C:\Users\skapuganti1\AppData\Local\anaconda3\Lib\site-packages\pybaseball\statcast.py:85: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  final_data = pd.concat(dataframe_list, axis=0).convert_dtypes(convert_string=False)


,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,player_name,batter,pitcher,events,description,...,batter_days_until_next_game,api_break_z_with_gravity,api_break_x_arm,api_break_x_batter_in,arm_angle,attack_angle,attack_direction,swing_path_tilt,intercept_ball_minus_batter_pos_x_inches,intercept_ball_minus_batter_pos_y_inches
160,KC,2024-10-30,77.5,-1.11,5.65,"Buehler, Walker",657077,621111,strikeout,swinging_strike_blocked,...,<NA>,5.23,-1.08,1.08,53.2,25.834292,-25.475081,32.831211,34.481443,57.315365
174,KC,2024-10-30,78.7,-1.01,5.73,"Buehler, Walker",657077,621111,None,swinging_strike,...,<NA>,5.28,-1.05,1.05,54.2,35.519261,-41.027263,35.112657,29.474286,57.624781
182,FC,2024-10-30,93.1,-1.19,5.53,"Buehler, Walker",657077,621111,None,swinging_strike,...,<NA>,1.89,-0.53,0.53,44.8,19.401316,-32.989729,26.71055,16.32129,37.919472
187,KC,2024-10-30,78.5,-1.19,5.7,"Buehler, Walker",657077,621111,None,ball,...,<NA>,5.16,-1.05,1.05,51.9,<NA>,<NA>,<NA>,<NA>,<NA>
204,KC,2024-10-30,77.4,-1.23,5.78,"Buehler, Walker",669224,621111,strikeout,swinging_strike,...,<NA>,5.2,-1.08,1.08,50.0,22.6438,-12.035832,32.683497,40.578398,40.302028


In [3]:
#To get a sense of the data, I wanted to see the size - pretty similar to 2023
df.shape

(760248, 118)

In [4]:
#To get the Batter names... the main dataset just has their ids
batter_ids = df['batter'].unique().tolist()
player_batter_names  = playerid_reverse_lookup(batter_ids, key_type = 'mlbam')
player_batter_names['batter_name'] = player_batter_names['name_last'].str.capitalize() + ', '+ player_batter_names['name_first'].str.capitalize()
player_batter_names.head()

,name_last,name_first,key_mlbam,key_retro,key_bbref,key_fangraphs,mlb_played_first,mlb_played_last,batter_name
0,miller,owen,680911,millo002,milleow01,24655,2021.0,2025.0,"Miller, Owen"
1,edman,tommy,669242,edmat001,edmanto01,19470,2019.0,2025.0,"Edman, Tommy"
2,castillo,diego,660636,castd004,castidi02,19906,2022.0,2024.0,"Castillo, Diego"
3,schneider,davis,676914,schnd001,schneda03,23565,2023.0,2026.0,"Schneider, Davis"
4,durbin,caleb,702332,durbc002,durbica01,29646,2025.0,2026.0,"Durbin, Caleb"


In [5]:
#Merging the batter information into the main dataset
df_player = pd.merge(
    df
    , player_batter_names [['key_mlbam', 'batter_name']]
    , left_on = 'batter'
    , right_on = 'key_mlbam'
    , how = 'inner'
).drop(columns = ['key_mlbam'])
df_player.head()

,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,player_name,batter,pitcher,events,description,...,api_break_z_with_gravity,api_break_x_arm,api_break_x_batter_in,arm_angle,attack_angle,attack_direction,swing_path_tilt,intercept_ball_minus_batter_pos_x_inches,intercept_ball_minus_batter_pos_y_inches,batter_name
0,KC,2024-10-30,77.5,-1.11,5.65,"Buehler, Walker",657077,621111,strikeout,swinging_strike_blocked,...,5.23,-1.08,1.08,53.2,25.834292,-25.475081,32.831211,34.481443,57.315365,"Verdugo, Alex"
1,KC,2024-10-30,78.7,-1.01,5.73,"Buehler, Walker",657077,621111,None,swinging_strike,...,5.28,-1.05,1.05,54.2,35.519261,-41.027263,35.112657,29.474286,57.624781,"Verdugo, Alex"
2,FC,2024-10-30,93.1,-1.19,5.53,"Buehler, Walker",657077,621111,None,swinging_strike,...,1.89,-0.53,0.53,44.8,19.401316,-32.989729,26.71055,16.32129,37.919472,"Verdugo, Alex"
3,KC,2024-10-30,78.5,-1.19,5.7,"Buehler, Walker",657077,621111,None,ball,...,5.16,-1.05,1.05,51.9,<NA>,<NA>,<NA>,<NA>,<NA>,"Verdugo, Alex"
4,KC,2024-10-30,77.4,-1.23,5.78,"Buehler, Walker",669224,621111,strikeout,swinging_strike,...,5.2,-1.08,1.08,50.0,22.6438,-12.035832,32.683497,40.578398,40.302028,"Wells, Austin"


In [6]:
#Doing the same for the pitcher information - getting their ids and names
pitcher_ids = df['pitcher'].unique().tolist()
player_pitcher_names = playerid_reverse_lookup(pitcher_ids, key_type = 'mlbam')
player_pitcher_names['pitcher_name'] = player_pitcher_names['name_first'].str.capitalize() + ', ' + player_pitcher_names['name_last'].str.capitalize()
player_pitcher_names.head()

,name_last,name_first,key_mlbam,key_retro,key_bbref,key_fangraphs,mlb_played_first,mlb_played_last,pitcher_name
0,miller,owen,680911,millo002,milleow01,24655,2021.0,2025.0,"Owen, Miller"
1,meeker,james,703231,meekj001,meekeja01,29891,2024.0,2024.0,"James, Meeker"
2,cessa,luis,570666,cessl001,cessalu01,13345,2016.0,2023.0,"Luis, Cessa"
3,zastryzny,rob,642239,zastr001,zastrro01,15094,2016.0,2025.0,"Rob, Zastryzny"
4,díaz,edwin,621242,diaze006,diazed04,14710,2016.0,2026.0,"Edwin, Díaz"


In [7]:
# Merging the pitcher information into the main dataset - similar to the batter data
df_player = pd.merge(
    df_player
    , player_pitcher_names[['key_mlbam', 'pitcher_name']]
    , left_on = 'pitcher'
    , right_on = 'key_mlbam'
    , how = 'inner'
).drop(columns = ['key_mlbam'])
df_player.head()

,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,player_name,batter,pitcher,events,description,...,api_break_x_arm,api_break_x_batter_in,arm_angle,attack_angle,attack_direction,swing_path_tilt,intercept_ball_minus_batter_pos_x_inches,intercept_ball_minus_batter_pos_y_inches,batter_name,pitcher_name
0,KC,2024-10-30,77.5,-1.11,5.65,"Buehler, Walker",657077,621111,strikeout,swinging_strike_blocked,...,-1.08,1.08,53.2,25.834292,-25.475081,32.831211,34.481443,57.315365,"Verdugo, Alex","Walker, Buehler"
1,KC,2024-10-30,78.7,-1.01,5.73,"Buehler, Walker",657077,621111,None,swinging_strike,...,-1.05,1.05,54.2,35.519261,-41.027263,35.112657,29.474286,57.624781,"Verdugo, Alex","Walker, Buehler"
2,FC,2024-10-30,93.1,-1.19,5.53,"Buehler, Walker",657077,621111,None,swinging_strike,...,-0.53,0.53,44.8,19.401316,-32.989729,26.71055,16.32129,37.919472,"Verdugo, Alex","Walker, Buehler"
3,KC,2024-10-30,78.5,-1.19,5.7,"Buehler, Walker",657077,621111,None,ball,...,-1.05,1.05,51.9,<NA>,<NA>,<NA>,<NA>,<NA>,"Verdugo, Alex","Walker, Buehler"
4,KC,2024-10-30,77.4,-1.23,5.78,"Buehler, Walker",669224,621111,strikeout,swinging_strike,...,-1.08,1.08,50.0,22.6438,-12.035832,32.683497,40.578398,40.302028,"Wells, Austin","Walker, Buehler"


In [8]:
#Realized that the player name in the main dataset is actually the pitcher name, and the name was ambiguous. I just deleted the player_name column
df_player = df_player.drop(columns = ['player_name'])
df_player

,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,batter,pitcher,events,description,spin_dir,...,api_break_x_arm,api_break_x_batter_in,arm_angle,attack_angle,attack_direction,swing_path_tilt,intercept_ball_minus_batter_pos_x_inches,intercept_ball_minus_batter_pos_y_inches,batter_name,pitcher_name
0,KC,2024-10-30,77.5,-1.11,5.65,657077,621111,strikeout,swinging_strike_blocked,<NA>,...,-1.08,1.08,53.2,25.834292,-25.475081,32.831211,34.481443,57.315365,"Verdugo, Alex","Walker, Buehler"
1,KC,2024-10-30,78.7,-1.01,5.73,657077,621111,None,swinging_strike,<NA>,...,-1.05,1.05,54.2,35.519261,-41.027263,35.112657,29.474286,57.624781,"Verdugo, Alex","Walker, Buehler"
2,FC,2024-10-30,93.1,-1.19,5.53,657077,621111,None,swinging_strike,<NA>,...,-0.53,0.53,44.8,19.401316,-32.989729,26.71055,16.32129,37.919472,"Verdugo, Alex","Walker, Buehler"
3,KC,2024-10-30,78.5,-1.19,5.7,657077,621111,None,ball,<NA>,...,-1.05,1.05,51.9,<NA>,<NA>,<NA>,<NA>,<NA>,"Verdugo, Alex","Walker, Buehler"
4,KC,2024-10-30,77.4,-1.23,5.78,669224,621111,strikeout,swinging_strike,<NA>,...,-1.08,1.08,50.0,22.6438,-12.035832,32.683497,40.578398,40.302028,"Wells, Austin","Walker, Buehler"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
752464,None,2024-03-15,<NA>,<NA>,<NA>,666310,663903,None,swinging_strike,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,"Naylor, Bo","Brady, Singer"
752465,None,2024-03-15,<NA>,<NA>,<NA>,666310,663903,None,swinging_strike,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,"Naylor, Bo","Brady, Singer"
752466,None,2024-03-15,<NA>,<NA>,<NA>,608070,663903,double,hit_into_play,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,"Ramírez, José","Brady, Singer"
752467,None,2024-03-15,<NA>,<NA>,<NA>,665926,663903,field_out,hit_into_play,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,"Giménez, Andrés","Brady, Singer"


In [9]:
#Saving the dataset as a csv...
df_player.to_csv("player_dataset_2024.csv", index = False)